# 06 — Norm Growth in Lattice-Based Folding (LatticeFold)

## The Problem

In lattice-based folding (LatticeFold), the witness consists of **short vectors** over a polynomial ring $R_q = \mathbb{Z}_q[X]/(X^n+1)$. The binding property of lattice commitments depends on these vectors staying short.

Each fold step: $\mathbf{W}' = \mathbf{W}_1 + r \cdot \mathbf{W}_2$

With naive folding, $\|\mathbf{W}'\| \approx \|r\| \cdot \|\mathbf{W}_2\|$, leading to **exponential norm growth** over many folds.

LatticeFold's solution: **decompose-then-fold**. Decompose witnesses into base-$B$ digits (each in $[0, B-1]$), fold the digits, then re-decompose. This bounds the intermediate norms.

## This Notebook

We run Monte Carlo simulations to empirically characterize:
1. How norm grows for naive vs decompose-then-fold
2. How parameter choices ($n$, $q$, $B$) affect intermediate norms
3. Whether theoretical worst-case bounds are tight

**LatticeFold paper:** eprint 2024/257

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from experiments.norm_growth.ring_arithmetic import RingParams
from experiments.norm_growth.lattice_fold_sim import FoldingConfig, simulate_folding

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## 1. Single-Trial Trajectory: Naive vs Decompose

First, let's see one trial of 200 folds to understand the qualitative behavior.

In [ ]:
params = RingParams(n=64, q=2**16)
N_FOLDS = 200

configs = {
    'Naive': FoldingConfig(params=params, base=0, witness_length=4, initial_bound=16),
    'B=2':   FoldingConfig(params=params, base=2, witness_length=4, initial_bound=16),
    'B=4':   FoldingConfig(params=params, base=4, witness_length=4, initial_bound=16),
    'B=8':   FoldingConfig(params=params, base=8, witness_length=4, initial_bound=16),
}

trajectories = {}
for name, cfg in configs.items():
    results = simulate_folding(cfg, num_folds=N_FOLDS, seed=42)
    trajectories[name] = {
        'inter_l2': [r.intermediate_digit_l2 for r in results],
        'inter_linf': [r.intermediate_digit_linf for r in results],
        'recomp_l2': [r.recomposed_l2 for r in results],
    }
    print(f"{name:>6}: final inter_L2={results[-1].intermediate_digit_l2:.1f}, "
          f"inter_Linf={results[-1].intermediate_digit_linf}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
steps = np.arange(1, N_FOLDS + 1)
colors = {'Naive': '#dc2626', 'B=2': '#2563eb', 'B=4': '#16a34a', 'B=8': '#9333ea'}

# Plot 1: Intermediate digit L2 norm (log scale)
for name in configs:
    axes[0].semilogy(steps, trajectories[name]['inter_l2'], 
                     color=colors[name], label=name, linewidth=1.5, alpha=0.8)
axes[0].set_xlabel('Fold step')
axes[0].set_ylabel('Intermediate digit L2 norm (log)')
axes[0].set_title('Intermediate Digit Norms')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Intermediate digit Linf norm
for name in configs:
    axes[1].semilogy(steps, trajectories[name]['inter_linf'],
                     color=colors[name], label=name, linewidth=1.5, alpha=0.8)
axes[1].set_xlabel('Fold step')
axes[1].set_ylabel('Intermediate digit L∞ norm (log)')
axes[1].set_title('Max Coefficient After Fold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Recomposed witness L2 (same for all — algebraic value)
for name in configs:
    axes[2].semilogy(steps, trajectories[name]['recomp_l2'],
                     color=colors[name], label=name, linewidth=1.5, alpha=0.8)
axes[2].set_xlabel('Fold step')
axes[2].set_ylabel('Recomposed witness L2 norm (log)')
axes[2].set_title('Algebraic Witness Norm (same for all)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Norm Evolution Over {N_FOLDS} Folds (n={params.n}, q=2^16)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('norm_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Monte Carlo: Distribution of Norms

A single trial shows the qualitative behavior. Now let's run many trials to characterize the **distribution** of norms. The question: what does the 99th percentile look like vs the theoretical worst case?

In [ ]:
N_TRIALS = 200
N_FOLDS_MC = 100

def run_mc(config, n_folds, n_trials):
    """Run Monte Carlo trials, return final intermediate L2 norms."""
    final_norms = []
    for trial in range(n_trials):
        results = simulate_folding(config, num_folds=n_folds, seed=trial * 7919 + 1)
        final_norms.append(results[-1].intermediate_digit_l2)
    return np.array(final_norms)

mc_results = {}
for name, cfg in configs.items():
    norms = run_mc(cfg, N_FOLDS_MC, N_TRIALS)
    mc_results[name] = norms
    print(f"{name:>6}: mean={np.mean(norms):.1f}, std={np.std(norms):.1f}, "
          f"p95={np.percentile(norms, 95):.1f}, p99={np.percentile(norms, 99):.1f}, "
          f"max={np.max(norms):.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograms of final intermediate L2 norms
for name in ['Naive', 'B=4']:
    axes[0].hist(mc_results[name], bins=30, alpha=0.6, color=colors[name], label=name, density=True)
axes[0].set_xlabel('Final intermediate digit L2 norm')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Distribution After {N_FOLDS_MC} Folds ({N_TRIALS} trials)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot comparing all strategies
data = [mc_results[name] for name in configs]
bp = axes[1].boxplot(data, labels=list(configs.keys()), patch_artist=True)
for patch, name in zip(bp['boxes'], configs):
    patch.set_facecolor(colors[name])
    patch.set_alpha(0.6)
axes[1].set_ylabel('Final intermediate digit L2 norm')
axes[1].set_title('Norm Distribution by Strategy')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('norm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Parameter Sweep: Ring Dimension

How does the ring dimension $n$ affect norm growth? Larger $n$ means longer polynomials and higher-dimensional vectors.

In [ ]:
ring_dims = [32, 64, 128, 256]
bases_to_test = [0, 2, 4, 8]  # 0 = naive
N_TRIALS_SWEEP = 50
N_FOLDS_SWEEP = 100

sweep_results = {}  # (n, base) -> mean final norm

for n in ring_dims:
    p = RingParams(n=n, q=2**16)
    for base in bases_to_test:
        cfg = FoldingConfig(params=p, base=base, witness_length=4, initial_bound=16)
        norms = run_mc(cfg, N_FOLDS_SWEEP, N_TRIALS_SWEEP)
        sweep_results[(n, base)] = {
            'mean': np.mean(norms),
            'p99': np.percentile(norms, 99),
        }
        label = f'B={base}' if base > 0 else 'Naive'
        print(f"n={n:>4}, {label:>6}: mean={np.mean(norms):>8.1f}, p99={np.percentile(norms, 99):>8.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
base_labels = {0: 'Naive', 2: 'B=2', 4: 'B=4', 8: 'B=8'}
base_colors = {0: '#dc2626', 2: '#2563eb', 4: '#16a34a', 8: '#9333ea'}

for base in bases_to_test:
    means = [sweep_results[(n, base)]['mean'] for n in ring_dims]
    p99s = [sweep_results[(n, base)]['p99'] for n in ring_dims]
    axes[0].semilogy(ring_dims, means, 'o-', color=base_colors[base], 
                     label=base_labels[base], linewidth=2, markersize=8)
    axes[1].semilogy(ring_dims, p99s, 'o-', color=base_colors[base],
                     label=base_labels[base], linewidth=2, markersize=8)

for ax, metric in zip(axes, ['Mean', '99th Percentile']):
    ax.set_xlabel('Ring dimension n')
    ax.set_ylabel(f'{metric} intermediate digit L2 norm')
    ax.set_title(f'{metric} Norm vs Ring Dimension (q=2^16, {N_FOLDS_SWEEP} folds)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(ring_dims)

plt.tight_layout()
plt.savefig('norm_vs_ring_dim.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Scaling Analysis: How Does Norm Grow with Fold Count?

The critical question: is the growth **linear**, **polynomial**, or **exponential** in the number of folds?

In [ ]:
params_scaling = RingParams(n=64, q=2**16)
fold_counts = [10, 25, 50, 100, 200, 500]
N_TRIALS_SCALE = 50

scaling_results = {}  # (base, n_folds) -> p99 norm

for base in [0, 2, 4]:
    for nf in fold_counts:
        cfg = FoldingConfig(params=params_scaling, base=base, witness_length=4, initial_bound=16)
        norms = run_mc(cfg, nf, N_TRIALS_SCALE)
        scaling_results[(base, nf)] = {
            'mean': np.mean(norms),
            'p99': np.percentile(norms, 99),
        }

fig, ax = plt.subplots(figsize=(10, 6))

for base in [0, 2, 4]:
    means = [scaling_results[(base, nf)]['mean'] for nf in fold_counts]
    label = f'B={base}' if base > 0 else 'Naive'
    color = base_colors[base]
    ax.loglog(fold_counts, means, 'o-', color=color, label=label, linewidth=2, markersize=8)

# Reference lines
ax.loglog(fold_counts, [fold_counts[0] * (nf/fold_counts[0])**0.5 * 50 for nf in fold_counts],
          '--', color='gray', alpha=0.5, label='O(sqrt(n)) reference')
ax.loglog(fold_counts, [fold_counts[0] * (nf/fold_counts[0]) * 10 for nf in fold_counts],
          ':', color='gray', alpha=0.5, label='O(n) reference')

ax.set_xlabel('Number of folds', fontsize=12)
ax.set_ylabel('Mean intermediate digit L2 norm', fontsize=12)
ax.set_title('Norm Scaling with Fold Count (n=64, q=2^16)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('norm_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

# Fit power law: norm ~ N^alpha
for base in [0, 2, 4]:
    means = [scaling_results[(base, nf)]['mean'] for nf in fold_counts]
    log_nf = np.log(fold_counts)
    log_means = np.log(means)
    alpha, _ = np.polyfit(log_nf, log_means, 1)
    label = f'B={base}' if base > 0 else 'Naive'
    print(f"{label}: norm ~ N^{alpha:.3f}")

## 5. Tightness of Theoretical Bounds

LatticeFold's security proof gives worst-case bounds on intermediate norms.
How tight are these bounds compared to typical-case behavior?

For a ternary challenge $r$ (coefficients in $\{-1, 0, 1\}$) and a digit vector with entries in $[0, B-1]$:
- Each coefficient of $r \cdot d$ is a sum of $\leq n$ products of $\{-1,0,1\} \times [0,B-1]$
- Worst case: $\|r \cdot d\|_\infty \leq n \cdot (B-1)$
- Typical case (CLT): $\|r \cdot d\|_\infty \approx O(\sqrt{n} \cdot B)$

The **tightness ratio** = empirical p99 / theoretical worst-case tells us how conservative the bounds are.

In [ ]:
# Tightness analysis for different parameters
print(f"{'n':>4} {'B':>4} {'Empirical p99 Linf':>20} {'Worst case n*(B-1)':>20} {'Tightness ratio':>16}")
print('-' * 70)

for n in [32, 64, 128, 256]:
    p = RingParams(n=n, q=2**16)
    for base in [2, 4, 8]:
        cfg = FoldingConfig(params=p, base=base, witness_length=4, initial_bound=16)
        linf_norms = []
        for trial in range(100):
            results = simulate_folding(cfg, num_folds=100, seed=trial * 7919 + 1)
            linf_norms.append(results[-1].intermediate_digit_linf)
        
        empirical_p99 = np.percentile(linf_norms, 99)
        worst_case = n * (base - 1)  # theoretical worst case for one ring mul
        ratio = empirical_p99 / worst_case if worst_case > 0 else 0
        
        print(f"{n:>4} {base:>4} {empirical_p99:>20.1f} {worst_case:>20} {ratio:>16.4f}")

## Key Findings

1. **Decompose-then-fold effectively bounds norm growth**: naive folding shows monotonic growth, while decomposition keeps intermediate norms bounded regardless of fold count.

2. **Smaller B = tighter norms**: B=2 gives the tightest bound but requires more digit vectors (higher space cost). B=4 and B=8 are practical trade-offs.

3. **Typical case is much better than worst case**: the tightness ratio shows that empirical 99th percentile norms are a small fraction of the theoretical worst-case bound. This suggests room for tighter security proofs.

4. **Norm scaling with fold count**: for decompose-then-fold, norms grow sub-linearly with fold count (roughly $\sqrt{N}$ for the recomposed witness, bounded for digit norms).

5. **Ring dimension dependence**: norms scale as $\sim \sqrt{n}$ with ring dimension, consistent with CLT-like concentration of random polynomial products.

## Implications for Parameter Selection

- The gap between typical and worst-case norms means **LatticeFold parameters could be made more aggressive** (smaller q, larger B) while maintaining practical security.
- For long IVC chains (thousands of folds), decomposition with B=2 or B=4 keeps norms well within bounds.
- The $\sqrt{n}$ dependence on ring dimension suggests that **larger rings don't incur disproportionate norm costs**.